In [5]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16, ResNet50
import numpy as np
import matplotlib.pyplot as plt

In [7]:
IMAGE_SIZE = (224, 224) # Standard size for VGG and ResNet
BATCH_SIZE = 32
NUM_CLASSES = 5 # Example: Healthy, Rust, Septoria, Mildew, Scab
DATASET_PATH = 'images'

In [8]:
# --- Data Augmentation and Preprocessing ---

# 1. Training Data Generator (with Augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,             # Normalize pixel values (0-1)
    shear_range=0.2,            # Randomly shear images
    zoom_range=0.2,             # Randomly zoom images
    horizontal_flip=True,       # Randomly flip images
    validation_split=0.2        # Set aside 20% for validation
)

In [9]:
# 2. Testing Data Generator (only Rescaling/Normalization)
test_datagen = ImageDataGenerator(rescale=1./255)

# --- Load Data from Directory ---

# Training Generator
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical', # Used for multi-class classification
    subset='training'
)

Found 4596 images belonging to 5 classes.


In [10]:
# Validation Generator (used for model selection during training)
validation_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 1148 images belonging to 5 classes.


In [24]:
def create_custom_cnn(input_shape, num_classes):
    model = Sequential([
        # First Block
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Second Block
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Dropout(0.25),

        # Classifier
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ], name="Custom_CNN")
    
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# cnn_model = create_custom_cnn((*IMAGE_SIZE, 3), NUM_CLASSES)

In [19]:
def create_vgg16_transfer_learning(input_shape, num_classes):
    # Load pre-trained VGG16 model (excluding the top classification layers)
    vgg_base = VGG16(weights='imagenet', 
                     include_top=False, 
                     input_shape=input_shape)

    # Freeze the layers of the VGG base model
    for layer in vgg_base.layers:
        layer.trainable = False

    # Build the new classification model on top of VGG base
    model = Sequential([
        vgg_base,
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ], name="VGG16_Transfer")

    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# vgg_model = create_vgg16_transfer_learning((*IMAGE_SIZE, 3), NUM_CLASSES)

In [11]:
def create_resnet50_transfer_learning(input_shape, num_classes):
    # Load pre-trained ResNet50 model (excluding the top classification layers)
    resnet_base = ResNet50(weights='imagenet', 
                           include_top=False, 
                           input_shape=input_shape)

    # Freeze the layers
    for layer in resnet_base.layers:
        layer.trainable = False

    # Build the new classification model on top of ResNet base
    x = resnet_base.output
    x = GlobalAveragePooling2D()(x) # Better than Flatten for deep models
    x = Dense(1024, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=resnet_base.input, outputs=predictions, name="ResNet50_Transfer")

    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# resnet_model = create_resnet50_transfer_learning((*IMAGE_SIZE, 3), NUM_CLASSES)

In [21]:
# --- Instantiate the Chosen Model ---
final_model = create_resnet50_transfer_learning((*IMAGE_SIZE, 3), NUM_CLASSES)
print(final_model.summary())

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 12s 0us/step


Model: "ResNet50_Transfer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 25,691,013 (98.00 MB)

 Trainable params: 2,103,301 (8.02 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

None


In [23]:
# --- Training Parameters ---
EPOCHS = 15 # Start with 10, adjust based on convergence

# --- Training ---
history = final_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // BATCH_SIZE
)

# --- Save the Trained Model ---
final_model.save('best_wheat_disease_resnet50.h5')
print("\nModel saved successfully!")

Epoch 1/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 386s 3s/step - accuracy: 0.2241 - loss: 1.7235 - val_accuracy: 0.2509 - val_loss: 1.5451
Epoch 2/15
  1/143 ━━━━━━━━━━━━━━━━━━━━ 4:17 2s/step - accuracy: 0.2500 - loss: 1.6201

d:\WDD\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


143/143 ━━━━━━━━━━━━━━━━━━━━ 86s 592ms/step - accuracy: 0.2500 - loss: 1.6201 - val_accuracy: 0.2696 - val_loss: 1.5407
Epoch 3/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 479s 3s/step - accuracy: 0.2844 - loss: 1.5761 - val_accuracy: 0.3196 - val_loss: 1.5357
Epoch 4/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 100s 689ms/step - accuracy: 0.2500 - loss: 1.6795 - val_accuracy: 0.2920 - val_loss: 1.5376
Epoch 5/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 530s 4s/step - accuracy: 0.3282 - loss: 1.5249 - val_accuracy: 0.3482 - val_loss: 1.4896
Epoch 6/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 100s 685ms/step - accuracy: 0.2188 - loss: 1.6207 - val_accuracy: 0.3571 - val_loss: 1.4892
Epoch 7/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 519s 4s/step - accuracy: 0.3596 - loss: 1.4816 - val_accuracy: 0.3732 - val_loss: 1.4824
Epoch 8/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 77s 522ms/step - accuracy: 0.3125 - loss: 1.5349 - val_accuracy: 0.3946 - val_loss: 1.4816
Epoch 9/15
143/143 ━━━━━━━━━━━━━━━━━━━━ 431s 3s/step - accuracy: 0.3902 - loss: 1.4561 - val_accur


Model saved successfully!


In [26]:
# --- Evaluation on Validation Set ---
print("\nEvaluating the final model...")

# The 'final_model' (which you trained and saved) is used here.
# 'validation_generator' contains the data held back for testing performance.
loss, accuracy = final_model.evaluate(
    validation_generator, 
    # Ensure steps cover all validation samples
    steps=validation_generator.samples // BATCH_SIZE 
)

print(f"\n✅ Final Model Test Accuracy ({final_model.name} Model): {accuracy * 100:.2f}%")
print(f"❌ Final Model Test Loss: {loss:.4f}")


Evaluating the final model...
35/35 ━━━━━━━━━━━━━━━━━━━━ 88s 3s/step - accuracy: 0.4241 - loss: 1.4079

✅ Final Model Test Accuracy (ResNet50_Transfer Model): 42.41%
❌ Final Model Test Loss: 1.4079


In [28]:
# --- Prediction Example ---
# To make a prediction on a new image:
# 1. Load and preprocess the image.
new_image = tf.keras.utils.load_img('images/Mildew/mildew_0.png', target_size=IMAGE_SIZE)
new_image_array = tf.keras.utils.img_to_array(new_image)
new_image_array = np.expand_dims(new_image_array / 255.0, axis=0) # Normalize and add batch dimension

# 2. Predict the class
predictions = final_model.predict(new_image_array)
predicted_class_index = np.argmax(predictions[0])
class_labels = list(train_generator.class_indices.keys())
print(f"Predicted Disease: {class_labels[predicted_class_index]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
Predicted Disease: Mildew
